In [1]:
from pathlib import Path

BASE = Path.cwd()
DATA = BASE / "data"
RAW = DATA / "raw"
INTERIM = DATA / "interim"

for p in [DATA, RAW, INTERIM]:
    p.mkdir(parents=True, exist_ok=True)

print("Folders created.")

Folders created.


In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
needed_clinvar_columns = [
    "VariationID",
    "GeneSymbol",
    "Type",
    "Assembly",
    "OriginSimple",
    "ClinicalSignificance",
    "ReviewStatus",
    "Chromosome",
    "Start",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "Name"
]

In [4]:
clinvar_path = INTERIM / "clinvar.csv"

clinvar = pd.read_csv(
    clinvar_path,
    low_memory=False,
)   
print(clinvar.shape)
clinvar.head(3)

(10310, 45)


,Unnamed: 0,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,ReferenceAllele,AlternateAllele,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,label
0,547,15323,single nucleotide variant,NM_000552.5(VWF):c.4883T>C (p.Ile1628Thr),7450,VWF,HGNC:12726,Pathogenic,1,"Aug 13, 2024",61750584,-,RCV000000308|RCV000086808|RCV000778377|RCV002243602|RCV004739270|RCV005237342|RCV005859448,"MONDO:MONDO:0015628,MedGen:C1282968,Orphanet:166084|MedGen:C3661900|MONDO:MONDO:0019565,MeSH:D014842,MedGen:C5703318,Orphanet:903|MONDO:MONDO:0013304,MedGen:C1264040,OMIM:613554,Orphanet:166081,Or...",Von Willebrand disease type 2A|not provided|Hereditary von Willebrand disease|von Willebrand disease type 2|VWF-related disorder|von Willebrand disorder|von Willebrand disease type 1,germline;unknown,germline,GRCh38,NC_000012.12,12,6018535,6018535,na,na,12p13.31,reviewed by expert panel,16,-,N,"ClinGen:CA114115,UniProtKB:P04275#VAR_005817,OMIM:613160.0001",3,284,6018535,A,G,-,-,-,-,-,-,SCV005442523,-,-,Pathogenic
1,549,15324,single nucleotide variant,NM_000552.5(VWF):c.4789C>T (p.Arg1597Trp),7450,VWF,HGNC:12726,Pathogenic,1,"Aug 13, 2024",61750117,-,RCV000000309|RCV000086797|RCV000623564|RCV000851942|RCV002243603,"MONDO:MONDO:0015628,MedGen:C1282968,Orphanet:166084|MedGen:C3661900|MeSH:D030342,MedGen:C0950123|MONDO:MONDO:0019565,MeSH:D014842,MedGen:C5703318,Orphanet:903|MONDO:MONDO:0013304,MedGen:C1264040,O...",Von Willebrand disease type 2A|not provided|Inborn genetic diseases|Hereditary von Willebrand disease|von Willebrand disease type 2,germline;unknown,germline,GRCh38,NC_000012.12,12,6018629,6018629,na,na,12p13.31,reviewed by expert panel,15,-,N,"ClinGen:CA114117,UniProtKB:P04275#VAR_005813,OMIM:613160.0002",3,285,6018629,G,A,-,-,-,-,-,-,SCV005442547,-,-,Pathogenic
2,553,15326,single nucleotide variant,NM_000552.5(VWF):c.3939G>C (p.Trp1313Cys),7450,VWF,HGNC:12726,Likely pathogenic,1,"Apr 01, 2025",61749392,-,RCV000000311|RCV000086709,"MONDO:MONDO:0015629,MedGen:C1282971,Orphanet:166087|MedGen:C3661900",Von Willebrand disease type 2B|not provided,germline,germline,GRCh38,NC_000012.12,12,6019479,6019479,na,na,12p13.31,reviewed by expert panel,3,-,N,"ClinGen:CA114121,UniProtKB:P04275#VAR_005796,OMIM:613160.0004",3,287,6019479,C,G,-,-,-,-,-,-,SCV006550592,-,-,Pathogenic


In [17]:
clin_nsfp_path = INTERIM / "clin_nsfp"
with open(clin_nsfp_path, 'r') as f:
    for _ in range(5):
        print(f.readline())

#chr	pos(1-based)	ref	alt	aaref	aaalt	rs_dbSNP	hg19_chr	hg19_pos(1-based)	hg18_chr	hg18_pos(1-based)	hs1_chr	hs1_pos(1-based)	aapos	genename	Ensembl_geneid	Ensembl_transcriptid	Ensembl_proteinid	Uniprot_acc	Uniprot_entry	HGVSc_snpEff	HGVSp_snpEff	HGVSc_VEP	HGVSp_VEP	APPRIS	GENCODE_basic	TSL	VEP_canonical	MANE	cds_strand	refcodon	codonpos	codon_degeneracy	Ancestral_allele	AltaiNeandertal	Denisova	VindijiaNeandertal	ChagyrskayaNeandertal	clinvar_id	clinvar_clnsig	clinvar_trait	clinvar_review	clinvar_hgvs	clinvar_var_source	clinvar_MedGen_id	clinvar_OMIM_id	clinvar_Orphanet_id	Interpro_domain	SIFT_score	SIFT_converted_rankscore	SIFT_pred	SIFT4G_score	SIFT4G_converted_rankscore	SIFT4G_pred	Polyphen2_HDIV_score	Polyphen2_HDIV_rankscore	Polyphen2_HDIV_pred	Polyphen2_HVAR_score	Polyphen2_HVAR_rankscore	Polyphen2_HVAR_pred	MutationTaster_score	MutationTaster_rankscore	MutationTaster_pred	MutationTaster_model	MutationTaster_trees_benign	MutationTaster_trees_deleterious	MutationAssessor_score	Mu

In [20]:
clin_nsfp = pd.read_csv(clin_nsfp_path, sep='\t', encoding='ISO-8859-1',low_memory=False)

In [21]:
clin_nsfp.head()

,#chr,pos(1-based),ref,alt,aaref,aaalt,rs_dbSNP,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),hs1_chr,hs1_pos(1-based),aapos,genename,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,Uniprot_acc,Uniprot_entry,HGVSc_snpEff,HGVSp_snpEff,HGVSc_VEP,HGVSp_VEP,APPRIS,GENCODE_basic,TSL,VEP_canonical,MANE,cds_strand,refcodon,codonpos,codon_degeneracy,Ancestral_allele,AltaiNeandertal,Denisova,VindijiaNeandertal,ChagyrskayaNeandertal,clinvar_id,clinvar_clnsig,clinvar_trait,clinvar_review,clinvar_hgvs,clinvar_var_source,clinvar_MedGen_id,clinvar_OMIM_id,clinvar_Orphanet_id,Interpro_domain,SIFT_score,SIFT_converted_rankscore,SIFT_pred,SIFT4G_score,SIFT4G_converted_rankscore,SIFT4G_pred,Polyphen2_HDIV_score,Polyphen2_HDIV_rankscore,Polyphen2_HDIV_pred,Polyphen2_HVAR_score,Polyphen2_HVAR_rankscore,Polyphen2_HVAR_pred,MutationTaster_score,MutationTaster_rankscore,MutationTaster_pred,MutationTaster_model,MutationTaster_trees_benign,MutationTaster_trees_deleterious,MutationAssessor_score,MutationAssessor_rankscore,MutationAssessor_pred,PROVEAN_score,PROVEAN_converted_rankscore,PROVEAN_pred,VEST4_score,VEST4_rankscore,MetaSVM_score,MetaSVM_rankscore,MetaSVM_pred,MetaLR_score,MetaLR_rankscore,MetaLR_pred,Reliability_index,MetaRNN_score,MetaRNN_rankscore,MetaRNN_pred,M-CAP_score,M-CAP_rankscore,M-CAP_pred,REVEL_score,REVEL_rankscore,MutPred2_score,MutPred2_rankscore,MutPred2_pred,MutPred2_top5_mechanisms,MVP_score,MVP_rankscore,gMVP_score,gMVP_rankscore,MisFit_D_score,MisFit_D_rankscore,MisFit_D_pred_lenient,...,ClinGen_Haploinsufficiency_PMID,ClinGen_Haploinsufficiency_Disease,P(rec),Known_rec_info,RVIS_EVS,RVIS_percentile_EVS,LoF-FDR_ExAC,RVIS_ExAC,RVIS_percentile_ExAC,ExAC_pLI,ExAC_pRec,ExAC_pNull,ExAC_nonTCGA_pLI,ExAC_nonTCGA_pRec,ExAC_nonTCGA_pNull,ExAC_nonpsych_pLI,ExAC_nonpsych_pRec,ExAC_nonpsych_pNull,gnomAD_pLI,gnomAD_pRec,gnomAD_pNull,gnomAD_lof.oe,gnomAD_mis.oe,gnomAD_LOEUF,gnomAD_MOEUF,ExAC_del.score,ExAC_dup.score,ExAC_cnv.score,ExAC_cnv_flag,GDI,GDI-Phred,Gene damage prediction (all disease-causing genes),Gene damage prediction (all Mendelian disease-causing genes),Gene damage prediction (Mendelian AD disease-causing genes),Gene damage prediction (Mendelian AR disease-causing genes),Gene damage prediction (all PID disease-causing genes),Gene damage prediction (PID AD disease-causing genes),Gene damage prediction (PID AR disease-causing genes),Gene damage prediction (all cancer disease-causing genes),Gene damage prediction (cancer recessive disease-causing genes),Gene damage prediction (cancer dominant disease-causing genes),LoFtool_score,Essential_gene,Essential_gene_CRISPR,Essential_gene_CRISPR2,Essential_gene_gene-trap,Gene_indispensability_score,Gene_indispensability_pred,Tissue_specificity(Uniprot),HPA_consensus_adipose_tissue,HPA_consensus_adrenal_gland,HPA_consensus_amygdala,HPA_consensus_appendix,HPA_consensus_basal_ganglia,HPA_consensus_bone_marrow,HPA_consensus_breast,HPA_consensus_cerebellum,HPA_consensus_cerebral_cortex,HPA_consensus_cervix,HPA_consensus_choroid_plexus,HPA_consensus_colon,HPA_consensus_duodenum,HPA_consensus_endometrium,HPA_consensus_epididymis,HPA_consensus_esophagus,HPA_consensus_fallopian_tube,HPA_consensus_gallbladder,HPA_consensus_heart_muscle,HPA_consensus_hippocampal_formation,HPA_consensus_hypothalamus,HPA_consensus_kidney,HPA_consensus_liver,HPA_consensus_lung,HPA_consensus_lymph_node,HPA_consensus_midbrain,HPA_consensus_ovary,HPA_consensus_pancreas,HPA_consensus_parathyroid_gland,HPA_consensus_pituitary_gland,HPA_consensus_placenta,HPA_consensus_prostate,HPA_consensus_rectum,HPA_consensus_retina,HPA_consensus_salivary_gland,HPA_consensus_seminal_vesicle,HPA_consensus_skeletal_muscle,HPA_consensus_skin,HPA_consensus_small_intestine,HPA_consensus_smooth_muscle,HPA_consensus_spinal_cord,HPA_consensus_spleen,HPA_consensus_stomach,HPA_consensus_testis,HPA_consensus_thymus,HPA_consensus_thyroid_gland,HPA_consensus_tongue,HPA_consensus_tonsil,HPA_consensus_urinary_bladder,HPA_consensus_vag

In [23]:
print(len(clin_nsfp.columns))
print(clin_nsfp.columns.tolist())

652
['#chr', 'pos(1-based)', 'ref', 'alt', 'aaref', 'aaalt', 'rs_dbSNP', 'hg19_chr', 'hg19_pos(1-based)', 'hg18_chr', 'hg18_pos(1-based)', 'hs1_chr', 'hs1_pos(1-based)', 'aapos', 'genename', 'Ensembl_geneid', 'Ensembl_transcriptid', 'Ensembl_proteinid', 'Uniprot_acc', 'Uniprot_entry', 'HGVSc_snpEff', 'HGVSp_snpEff', 'HGVSc_VEP', 'HGVSp_VEP', 'APPRIS', 'GENCODE_basic', 'TSL', 'VEP_canonical', 'MANE', 'cds_strand', 'refcodon', 'codonpos', 'codon_degeneracy', 'Ancestral_allele', 'AltaiNeandertal', 'Denisova', 'VindijiaNeandertal', 'ChagyrskayaNeandertal', 'clinvar_id', 'clinvar_clnsig', 'clinvar_trait', 'clinvar_review', 'clinvar_hgvs', 'clinvar_var_source', 'clinvar_MedGen_id', 'clinvar_OMIM_id', 'clinvar_Orphanet_id', 'Interpro_domain', 'SIFT_score', 'SIFT_converted_rankscore', 'SIFT_pred', 'SIFT4G_score', 'SIFT4G_converted_rankscore', 'SIFT4G_pred', 'Polyphen2_HDIV_score', 'Polyphen2_HDIV_rankscore', 'Polyphen2_HDIV_pred', 'Polyphen2_HVAR_score', 'Polyphen2_HVAR_rankscore', 'Polyphen2_

In [24]:
KEEP_COLS = [
    # Keys
    '#chr', 'pos(1-based)', 'ref', 'alt',

    # Sequence / protein
    'aaref', 'aaalt', 'aapos', 'genename',

    # Modern scores
    'REVEL_score',
    'AlphaMissense_score',
    'CADD_phred',
    'ClinPred_score',
    'BayesDel_addAF_score',

    # Supporting scores
    'MetaRNN_score',
    'M-CAP_score',
    'PrimateAI_score',

    # Conservation
    'GERP++_RS',
    'phyloP100way_vertebrate',
    'phastCons100way_vertebrate',

    # Population
    'gnomAD4.1_joint_AF',
    'gnomAD4.1_joint_POPMAX_AF',
    'dbNSFP_POPMAX_AF',

    # Classic
    'SIFT_score',
    'Polyphen2_HDIV_score'
]

In [25]:
existing = [c for c in KEEP_COLS if c in clin_nsfp.columns]
missing = [c for c in KEEP_COLS if c not in clin_nsfp.columns]

print("Existing:", existing)
print("Missing:", missing)

Existing: ['#chr', 'pos(1-based)', 'ref', 'alt', 'aaref', 'aaalt', 'aapos', 'genename', 'REVEL_score', 'AlphaMissense_score', 'CADD_phred', 'ClinPred_score', 'BayesDel_addAF_score', 'MetaRNN_score', 'M-CAP_score', 'PrimateAI_score', 'GERP++_RS', 'phyloP100way_vertebrate', 'phastCons100way_vertebrate', 'gnomAD4.1_joint_AF', 'gnomAD4.1_joint_POPMAX_AF', 'dbNSFP_POPMAX_AF', 'SIFT_score', 'Polyphen2_HDIV_score']
Missing: []


In [27]:
available_cols = [col for col in KEEP_COLS if col in clin_nsfp.columns]
clin_nsfp = clin_nsfp[available_cols]

In [29]:
clin_nsfp.shape

(6673, 24)

In [30]:
clin_nsfp = clin_nsfp.drop_duplicates()

In [31]:
clin_nsfp.shape

(6673, 24)

In [32]:
clin_nsfp = clin_nsfp.dropna(subset=['#chr', 'pos(1-based)', 'ref', 'alt'])

In [33]:
clin_nsfp.shape

(6673, 24)

In [34]:
clin_nsfp = clin_nsfp[
    (clin_nsfp['ref'].str.len() == 1) &
    (clin_nsfp['alt'].str.len() == 1)
]

In [35]:
clin_nsfp.shape

(6673, 24)

In [36]:
clin_nsfp.head()

,#chr,pos(1-based),ref,alt,aaref,aaalt,aapos,genename,REVEL_score,AlphaMissense_score,CADD_phred,ClinPred_score,BayesDel_addAF_score,MetaRNN_score,M-CAP_score,PrimateAI_score,GERP++_RS,phyloP100way_vertebrate,phastCons100way_vertebrate,gnomAD4.1_joint_AF,gnomAD4.1_joint_POPMAX_AF,dbNSFP_POPMAX_AF,SIFT_score,Polyphen2_HDIV_score
0,1,9710459,C,A,P,T,2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2;2,PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD,.;.;.;0.166;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.166;.,.;.;.;0.1775;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.2881;.,23.5,0.563564896583557,-0.310633,.;.;.;0.12227568;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.12227568;.,0.043748,0.711624264717,3.29,4.380,1.0,8.736350e-05,1.097910e-03,1.347e-03,.;.;.;0.004;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.003;.,.;0.971;.;0.971;.;.;0.971;0.662;0.971;0.971;0.971;0.971;0.971;0.662;0.971;.;.;.;0.971;0.662;.;.;.
1,1,9710567,C,T,R,C,38;38;38;38;38;38;38;38;38;38;38;38;38;38;38;38;38;38;38;38;38;38;38,PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD,.;.;.;0.238;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.238;.,.;.;.;0.0757;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.1332;.,21.5,0.54362165927887,-0.182523,.;.;.;0.13606924;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.13606924;.,0.02438,0.41577732563,3.56,2.906,1.0,2.292690e-05,3.207180e-04,9.411e-04,.;.;.;0.142;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.112;.,.;0.232;.;0.232;.;.;0.232;0.126;0.232;0.232;0.232;0.232;0.232;0.126;0.232;.;.;.;0.232;0.126;.;.;.
2,1,9715640,G,A,E,K,81;81;81;81;81;81;81;81;81;81;81;81;81;81;81;81;81;81;81;81;81;81,PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD,.;.;0.661;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.661;.,.;.;0.6296;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.6411;.,31,0.988526105880737,0.193758,.;.;0.7615843;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.7615843;.,0.074167,0.685891032219,5.95,9.914,1.0,6.196530e-07,8.474300e-07,1.206e-06,.;.;0.005;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.004;.,1.0;.;1.0;.;.;1.0;0.998;1.0;1.0;1.0;1.0;1.0;0.998;1.0;.;.;.;1.0;0.998;.;.;.
3,1,9715849,G,A,G,D,124;124;124;124;124;124;124;124;124;124;124;124;124;124;124;124;124;124;124;124;124;124,PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD,.;.;0.244;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.244;.,.;.;0.5383;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.7713;.,29.6,0.983129441738129,0.256222,.;.;0.7654717;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.7654717;.,0.126726,0.816306352615,5.57,9.481,1.0,.,.,.,.;.;0.062;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.058;.,1.0;.;1.0;.;.;1.0;0.823;1.0;1.0;1.0;1.0;1.0;0.823;1.0;.;.;.;1.0;0.823;.;.;.
4,1,9715914,T,A,F,I,146;146;146;146;146;146;146;146;146;146;146;146;146;146;146;146;146;146;146;146;146;146,PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD;PIK3CD,.;.;0.165;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.165;.,.;.;0.1385;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.3513;.,20.2,0.0272890378799048,-0.30888,.;.;0.010897398;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.010897398;.,0.016142,0.649348080158,4.37,5.766,1.0,9.245180e-04,5.942560e-03,5.943e-03,.;.;0.322;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;.;0.285;.,0.046;.;0.046;.;.;0.046;0.021;0.046;0.046;0.046;0.046;0.046;0.021;0.046;.;.;.;0.046;0.021;.;.;.


In [41]:
clin_nsfp.to_csv(f"{INTERIM}/nsfp_cleaned.csv", index=False)